# 🛠️ setup_colab.ipynb — Daily GitHub loop (Colab)


# 🔁 Daily Repo Loop for **Ego4d-LiteSTA** (Colab-first)

This block sets up SSH to GitHub (**Deploy Key stored in Drive**), pulls your repo, runs code, logs outputs, and pushes logs/artifacts back to GitHub.
- One-time: add your **Deploy Key** public key to your GitHub repo Deploy Keys with **write access**.
- Edit the variables (**GITHUB_USER**, **REPO_NAME**) below once; they persist per notebook.


In [3]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)


Mounted at /content/drive


In [4]:
GITHUB_USER = "saeedzns"
REPO_NAME   = "Ego4d-LiteSTA"
REPO_SSH    = f"git@github.com:{GITHUB_USER}/{REPO_NAME}.git"
SSH_DIR     = "/content/drive/MyDrive/.ssh"
WORKDIR     = "/content/drive/MyDrive/Ego4d-LiteSTA"


print("Repo:", REPO_SSH)
print("SSH dir:", SSH_DIR)
print("Work dir:", WORKDIR)


Repo: git@github.com:saeedzns/Ego4d-LiteSTA.git
SSH dir: /content/drive/MyDrive/.ssh
Work dir: /content/drive/MyDrive/Ego4d-LiteSTA


In [5]:
# 🔐 SSH + git config + clone/pull
# 1) Git identity (optional but recommended)
!git config --global user.name "saeedzns"
!git config --global user.email "zns1992@gmail.com"

In [6]:
# 2) SSH agent and key from Drive (key filename: colab_egolite)
%%bash
set -euo pipefail
SSH_DIR="/content/drive/MyDrive/.ssh"
KEY="$SSH_DIR/colab_egolite"

mkdir -p "$SSH_DIR"
chmod 700 "$SSH_DIR"

# Generate an ed25519 key if it doesn't exist
if [[ ! -f "$KEY" ]]; then
  ssh-keygen -t ed25519 -C "zns1992@gmail.com" -f "$KEY" -N ""
  echo "✅ New SSH key created at: $KEY"
else
  echo "ℹ️ SSH key already exists at: $KEY"
fi

chmod 600 "$KEY"
chmod 644 "$KEY.pub"

echo
echo "🔑 Add this PUBLIC key to your GitHub account (Settings → SSH and GPG keys → New SSH key):"
echo "-----------------------------------------------------------------------"
cat "$KEY.pub"
echo "-----------------------------------------------------------------------"
echo "Name it e.g.: colab_egolite"


ℹ️ SSH key already exists at: /content/drive/MyDrive/.ssh/colab_egolite

🔑 Add this PUBLIC key to your GitHub account (Settings → SSH and GPG keys → New SSH key):
-----------------------------------------------------------------------
ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIKJKrAL1TWfWxBlKM3xTWrTsDpTiT0spItDidzUDVbYa zns1992@gmail.com
-----------------------------------------------------------------------
Name it e.g.: colab_egolite


In [7]:
%%bash
set -euo pipefail

SSH_DIR="/content/drive/MyDrive/.ssh"
KEY="$SSH_DIR/colab_egolite"
REPO_SSH="git@github.com:saeedzns/Ego4d-LiteSTA.git"
WORKDIR="/content/drive/MyDrive/Ego4d-LiteSTA"

# SSH client config
mkdir -p ~/.ssh && chmod 700 ~/.ssh
cat > ~/.ssh/config <<EOF
Host github.com
  HostName github.com
  User git
  IdentityFile $KEY
  IdentitiesOnly yes
  StrictHostKeyChecking accept-new
EOF
chmod 600 ~/.ssh/config

# Agent + key
eval "$(ssh-agent -s)"
ssh-add "$KEY"

# Quick handshake test (should say "Hi <username>! You've successfully authenticated...")
ssh -T git@github.com || true

# Clone if missing; otherwise pull
if [[ ! -d "$WORKDIR/.git" ]]; then
  git clone "$REPO_SSH" "$WORKDIR"
else
  cd "$WORKDIR"
  git pull --rebase
fi

echo "✅ Repo ready at: $WORKDIR"


Agent pid 3466


Identity added: /content/drive/MyDrive/.ssh/colab_egolite (zns1992@gmail.com)
Hi saeedzns! You've successfully authenticated, but GitHub does not provide shell access.


In [ ]:
# 3) Clone or pull
import os

# Check if it's already a proper git repo
if not os.path.exists(f"{WORKDIR}/.git"):
    # If the directory exists but isn't a repo, remove it first
    if os.path.exists(WORKDIR):
        !rm -rf "$WORKDIR"
    !git clone "$REPO_SSH" "$WORKDIR"
else:
    %cd "$WORKDIR"
    !git pull --rebase

%cd "$WORKDIR"
!mkdir -p runs logs outputs notebooks
print("✅ Repo ready at", WORKDIR)


Cloning into '/content/drive/MyDrive/Ego4d-LiteSTA'...
remote: Enumerating objects: 14, done.
remote: Counting objects: 100% (14/14), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 14 (delta 3), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (14/14), 30.47 KiB | 3.39 MiB/s, done.
Resolving deltas: 100% (3/3), done.
/content/drive/MyDrive/Ego4d-LiteSTA
✅ Repo ready at /content/drive/MyDrive/Ego4d-LiteSTA


In [8]:
# 🧾 Run ID + logger helpers (use these in your train/eval cells)
import os, json, traceback, datetime, shutil, glob

RUN_ID = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
LOG_DIR = "logs"; RUN_DIR = f"runs/{RUN_ID}"; OUT_DIR = "outputs"
os.makedirs(LOG_DIR, exist_ok=True); os.makedirs(RUN_DIR, exist_ok=True); os.makedirs(OUT_DIR, exist_ok=True)

LOG_PATH = f"{LOG_DIR}/{RUN_ID}.log"
ERR_FILE = "COLAB_ERRORS.md"

def log_write(msg):
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(msg + "\n")
    print(msg)

def record_error(title, err_text):
    with open(ERR_FILE, "a", encoding="utf-8") as f:
        f.write(f"\n## {RUN_ID} — {title}\n\n```\n{err_text}\n```\n")

log_write(f"New run: {RUN_ID}")

New run: 20251030_205045


➡️ After your training/eval cells, run the next cell to push logs/outputs back:

In [9]:
# 🚀 Commit & push logs/artifacts
import json, glob, datetime
import os
%cd "$WORKDIR"

# Ensure output directory exists
os.makedirs("outputs", exist_ok=True)

# Optional: include metrics summary in the commit message if exists
metrics_candidates = sorted(glob.glob("runs/*/metrics.json"))
metrics_note = ""
if metrics_candidates:
    try:
        with open(metrics_candidates[-1]) as f:
            m = json.load(f)
        metrics_note = f" | sel_acc={m.get('selector_acc')} verb_acc={m.get('verb_acc')} ttc_mae={m.get('ttc_mae')}"
    except Exception as e:
        print("Could not read metrics.json:", e)

!git add logs/* runs/* outputs/* COLAB_ERRORS.md || true
!git status --porcelain

COMMIT_MSG = f"colab: logs & outputs {datetime.datetime.now().isoformat(timespec='seconds')}{metrics_note}"
# Use `git commit -m "..." || true` to avoid error if nothing to commit
!git commit -m "$COMMIT_MSG" || true
!git push
print("Pushed to GitHub.")

/content/drive/MyDrive/Ego4d-LiteSTA
fatal: pathspec 'outputs/*' did not match any files
 D Ego4D_STA_Colab_Starter.ipynb
 D Ego4D_STA_Full_Colab.ipynb
 D setup_colab.ipynb
?? COLAB_ERRORS.md
?? notebooks/
On branch main
Your branch is up to date with 'origin/main'.

Changes not staged for commit:
  (use "git add/rm <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	deleted:    Ego4D_STA_Colab_Starter.ipynb
	deleted:    Ego4D_STA_Full_Colab.ipynb
	deleted:    setup_colab.ipynb

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	COLAB_ERRORS.md
	notebooks/

no changes added to commit (use "git add" and/or "git commit -a")
Everything up-to-date
Pushed to GitHub.
